# Method 2 — Bi-Encoder (Kaggle T4)

Phase 2 của `docs/method2_plan.md`: train 2 vòng, pre-compute index, hiệu chỉnh ngưỡng. Ngân sách ~4h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [1]:
# ===== Cell 0: dò dataset + HF cache =====
# PHẢI chạy trước mọi import transformers: thư viện chốt cache lúc import,
# set HF_HOME sau đó thì không còn tác dụng.
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def _dirs_within(base: Path, max_depth: int = 4):
    """Mọi thư mục tới độ sâu `max_depth`, bỏ qua `hub/` cho nhanh."""
    frontier, seen = [base], []
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                children = [c for c in d.iterdir() if c.is_dir() and c.name != 'hub']
            except (PermissionError, OSError):
                continue
            seen.extend(children)
            nxt.extend(children)
        frontier = nxt
    return seen


def find_root(marker: str, label: str) -> Path:
    """Tìm thư mục chứa `marker`.

    Kaggle mount theo dạng /kaggle/input/datasets/<user>/<ds>/<ds>/, và số tầng
    đổi theo cách upload. Dò theo marker thì không phải hardcode username hay
    độ sâu — upload kiểu nào cũng tìm ra.
    """
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT):
        if (d / marker).exists():
            return d
    raise SystemExit(
        f'Không tìm thấy {label}: không thư mục nào dưới {INPUT_ROOT} có {marker}.\n'
        'Kiểm tra đã Add đủ 3 dataset ở sidebar Input chưa.'
    )


SRC_ROOT = find_root('src/models/preflight.py', 'dataset src')
DATA_ROOT = find_root('method2/manifest.json', 'dataset data')
HF_HOME = find_root('hub/models--BAAI--bge-m3', 'dataset hf-cache')

print('SRC :', SRC_ROOT)
print('DATA:', DATA_ROOT)
print('HF  :', HF_HOME)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# Có thư mục model chưa đủ — thiếu file trọng số thì lỗi chỉ lộ ra lúc nạp
# model, sau khi đã tốn thời gian cài đặt và copy.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    weights = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert weights, f'{name}: không có file trọng số > 100 MB'
    print(f'  {name}: {max(f.stat().st_size for f in weights) / 1024**3:.2f} GB')
print('\nHF cache OK')


SRC : /kaggle/input/datasets/dathq12/toolcalling-vi-src/toolcalling-vi-src
DATA: /kaggle/input/datasets/dathq12/toolcalling-vi-data/toolcalling-vi-data
HF  : /kaggle/input/datasets/dathq12/toolcalling-vi-hf-cache/toolcalling-vi-hf-cache
  models--BAAI--bge-m3: 2.12 GB
  models--xlm-roberta-base: 1.04 GB

HF cache OK


In [2]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# PEFT 0.20 raise nếu image có torchao < 0.16. Method 2 không dùng
# torchao quantization nên gỡ hẳn là xong.
!pip uninstall -y -q torchao 2>/dev/null || true

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
n_gpu = torch.cuda.device_count()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('số GPU:', n_gpu)

# sentence-transformers tự bọc DataParallel khi thấy >1 GPU. Với GradCache
# gọi model hàng trăm lần mỗi step thì phí đồng bộ cộng dồn rất nhanh.
if n_gpu > 1:
    print('  >1 GPU — truyền --single-gpu cho MỌI lệnh train')

# T4 là Turing (sm_75), KHÔNG có bf16 phần cứng. torch vẫn có thể báo
# is_bf16_supported()=True vì hỗ trợ qua emulation, chậm hơn fp16.
# Giữ fp16 bất kể giá trị này.
print('bf16 (emulated trên T4, vẫn dùng fp16):', torch.cuda.is_bf16_supported())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.7 MB/s eta 0:00:00
Tesla T4 14.5 / 14.6 GB free
số GPU: 2
  >1 GPU — truyền --single-gpu cho MỌI lệnh train
bf16 (emulated trên T4, vẫn dùng fp16): True


In [3]:
# ===== Cell 2: copy code + data vào /kaggle/working =====
# Dataset chỉ đọc, mà code ghi checkpoint và dùng đường dẫn tương đối, nên
# phải copy sang thư mục ghi được. Dùng path đã dò ở Cell 0.
import shutil

WORK = Path('/kaggle/working')
# `scripts` cần thiết: benchmark_biencoder.py chạy trên Kaggle.
for name in ('src', 'configs', 'scripts'):
    target = WORK / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(SRC_ROOT / name, target)

# Dataset data bắt đầu thẳng bằng method2/ custom_vi/ benchmark_vi/ (KHÔNG có
# tầng `data/`), còn code tham chiếu `data/method2/...` → copy vào data/.
data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir(parents=True)
for child in DATA_ROOT.iterdir():
    dest = data_dir / child.name
    shutil.copytree(child, dest) if child.is_dir() else shutil.copy2(child, dest)

%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0, kế thừa sang mọi tiến trình con `!python`.

print('src    :', sorted(p.name for p in (WORK / 'src').iterdir()))
print('data   :', sorted(p.name for p in data_dir.iterdir()))


/kaggle/working
src    : ['README.md', 'data', 'evaluation', 'models']
data   : ['benchmark_vi', 'custom_vi', 'method2']


In [4]:
# ===== Cell 3: kiểm tra bản copy TRƯỚC khi preflight =====
# Preflight kiểm tra tính đúng đắn của dữ liệu; cell này kiểm tra bước copy —
# tách ra để khi hỏng thì biết ngay là hỏng ở đâu.
REQUIRED = [
    'data/method2/decontamination.json',
    'data/method2/manifest.json',
    'data/method2/tool_pool.json',
    'data/method2/biencoder/train.jsonl',
    'data/method2/biencoder/val.jsonl',
    'data/method2/biencoder/pairs_stats.json',
    'data/method2/crossencoder/train.jsonl',
    'data/method2/crossencoder/val.jsonl',
    'data/method2/label_stats.json',
    'data/custom_vi/v1/test_seen.jsonl',
    'data/benchmark_vi/test.jsonl',
    'configs/method2/biencoder.yaml',
    'configs/method2/pinned_versions.json',
    'src/models/preflight.py',
]
missing = []
for rel in REQUIRED:
    path = WORK / rel
    if path.exists() and path.stat().st_size > 0:
        print(f'  {path.stat().st_size / 1024**2:8.2f} MB  {rel}')
    else:
        missing.append(rel)
        print(f'  {"THIẾU":>11}  {rel}')
assert not missing, f'Copy chưa đủ: {missing}'

# import được thì mới chắc src/ copy nguyên vẹn.
import importlib

importlib.import_module('src.models.preflight')
manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('\nsnapshot commit:', manifest.get('git_commit'))
print('copy OK')


     10.81 MB  data/method2/decontamination.json
      0.00 MB  data/method2/manifest.json
      5.01 MB  data/method2/tool_pool.json
     46.20 MB  data/method2/biencoder/train.jsonl
      5.47 MB  data/method2/biencoder/val.jsonl
      0.00 MB  data/method2/biencoder/pairs_stats.json
     91.13 MB  data/method2/crossencoder/train.jsonl
     11.29 MB  data/method2/crossencoder/val.jsonl
      0.00 MB  data/method2/label_stats.json
      5.18 MB  data/custom_vi/v1/test_seen.jsonl
     14.18 MB  data/benchmark_vi/test.jsonl
      0.00 MB  configs/method2/biencoder.yaml
      0.00 MB  configs/method2/pinned_versions.json
      0.01 MB  src/models/preflight.py

snapshot commit: 6eb1b582b4eaba9ff8a9b51c4d88ae5bd0898465
copy OK


## Pre-flight — cổng fail-closed TRƯỚC mọi training

```
decontamination.json tồn tại
        ↓
SHA-256 == manifest.json
        ↓
overlap train/val/test == 0
        ↓
unseen positive leakage == 0
        ↓
package versions khớp bản đã pin
        ↓
CHO PHÉP TRAIN
```

Thiếu file hoặc hash lệch → job dừng ngay, **không rebuild tự động**. Nếu
experiment chính tự dựng lại index từ dữ liệu đang có trên máy thì ta mất
đúng thứ cần đảm bảo: bằng chứng model được train trên đúng split đã kiểm
định. Rebuild là lệnh preprocessing riêng, chạy ở local rồi upload lại:
`python -m src.models.sources decontaminate && python -m src.models.sources manifest`

Vì sao `val ∩ test` là rủi ro nặng nhất: dù không train trên query đó, việc
chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan hơn
thực tế. `data/benchmark_vi` **giữ nguyên** — decontamination nằm ở tầng
dataset của Method 2 nên bốn method vẫn được đánh giá trên cùng một tập test.


In [5]:
# Exit code != 0 → dừng notebook, không chạy tiếp cell training nào.
!python -m src.models.preflight \
    --config configs/method2/biencoder.yaml \
    --require-gpu T4 \
    --output results/method2/preflight.json

preflight = json.load(open('results/method2/preflight.json', encoding='utf-8'))
assert preflight['passed'], f"Preflight KHÔNG ĐẠT: {preflight['failures']}"
print('preflight PASS —', len(preflight['checks']), 'check')


[PASS] commit SHA — 6eb1b582b4ea (từ manifest, không có .git)
[PASS] working tree sạch — không áp dụng — chạy từ snapshot
[PASS] config parse được — configs/method2/biencoder.yaml (0180b9e53df282b7…)
[PASS] manifest tồn tại — data/method2/manifest.json
[PASS] decontamination.json tồn tại — data/method2/decontamination.json
[PASS] SHA-256 data/method2/decontamination.json — 37bf70a801f7f2cf…
[PASS] SHA-256 data/method2/tool_pool.json — 4b3357fa13adbd8c…
[PASS] SHA-256 data/method2/biencoder/train.jsonl — eaf94362480fe1d6…
[PASS] SHA-256 data/method2/crossencoder/train.jsonl — 24d0ef10e3724e01…
[PASS] overlap Bi-Encoder == 0 — {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[PASS] overlap Cross-Encoder == 0 — {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[PASS] unseen positive leakage == 0 — 0 tool
[PASS] hai stage dùng chung index — khớp
[PASS] version transformers — 5.15.1
[PASS] version sentence-transformers — 6.0.0
[PASS] version peft — 0.20.0
[PASS] version torch (ghi nhận) — 2.

In [6]:
# Số liệu split để đối chiếu bằng mắt trước khi tiêu giờ GPU.
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])


unique query/split : {'test': 9331, 'train': 58829, 'val': 7819}
positive pairs     : 102100
negative samples   : 18334
query trùng split  : 1718 {'test∩train': 574, 'train∩val': 572, 'test∩train∩val': 542, 'test∩val': 30}
sample bị loại     : 32340 {'train->test': 26778, 'val->test': 3437, 'train->val': 2125}
overlap còn lại    : {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}


# Run 1a — Benchmark cấu hình (BẮT BUỘC trước smoke)

Lần chạy đầu trên T4 cho **475 s/step**: 100 step mất 13.2 giờ, một epoch
mất 49 giờ, trong khi plan dự toán 50-70 phút/epoch. Lệch ~45× nên phải tìm
cấu hình dùng được trước, đừng chạy tiếp smoke 100 step.

Vì sao mỗi step đắt: `CachedMNRL` không phải một forward/backward bình
thường. Effective batch 256, mỗi sample có anchor + positive + 4 negative →
**1,536 lượt encode**. Chia mini_batch 8 thành 192 chunk, GradCache chạy
**hai** pha (forward no-grad để cache, rồi forward+backward tính lại) →
~384 lần gọi model mỗi step. Mỗi lần chỉ 8×192 = 1,536 token, quá nhỏ để lấp
đầy T4 nên phần lớn thời gian là overhead — cộng thêm DataParallel giữa 2 GPU
thì nhân lên tiếp.

| Case | GPU | batch | mini | ckpt | đổi gì so với case trước |
|---|---|---|---|---|---|
| A | 1×T4 | 256 | 8 | on | tách ảnh hưởng DataParallel |
| B | 1×T4 | 256 | 16 | on | nửa số lần gọi model |
| C | 1×T4 | 256 | 32 | on | 1/4 số lần gọi model |
| D | 1×T4 | 128 | 32 | on | giảm effective batch |
| E | 1×T4 | 256 | 32 | off | tắt grad checkpointing |

A→C chỉ đổi **tốc độ**. D đổi **chất lượng**: MNRL mạnh lên theo số in-batch
negative, giảm batch là giảm negative — chỉ dùng khi A–C không đủ, và phải
ghi rõ vào báo cáo.


In [7]:
# Grid đã chạy ở session trước, case E thắng và đã nằm trong YAML. Chạy lại
# tốn ~35 phút (5 case × 5 step + 5 lần nạp BGE-M3) — chỉ bật khi đổi
# backbone, đổi max_seq_length, hoặc đổi n_hard_negatives.
RUN_BENCHMARK = False

# 5 step mỗi case, tắt eval (eval trên corpus 4,4k tool làm nhiễu số đo).
# `sec_per_step` lấy từ `train_runtime` của HF nên KHÔNG gồm thời gian nạp
# BGE-M3 — với run 5 step thì nạp model lấn át hoàn toàn wall-clock.
import subprocess, sys

if RUN_BENCHMARK:
    # subprocess thay vì `!` trong `if`: exit code hiện ra rõ ràng.
    subprocess.run([sys.executable, 'scripts/method2/benchmark_biencoder.py',
                    '--steps', '5',
                    '--config', 'configs/method2/biencoder.yaml',
                    '--output', 'results/method2/benchmark_biencoder.json'],
                   check=True)
else:
    print('BỎ QUA benchmark — cấu hình E (batch 256, mini 32, ckpt off) đã chốt.')


BỎ QUA benchmark — cấu hình E (batch 256, mini 32, ckpt off) đã chốt.


### Chốt cấu hình

Chọn case nhanh nhất mà VRAM còn an toàn (< ~13 GB để chừa chỗ cho eval),
rồi ghi vào `configs/method2/biencoder.yaml` trước khi chạy smoke.

Ngưỡng thực dụng: **> 60 s/step là chưa dùng được** — 370 step/epoch × 3
epoch mà 60 s/step đã là 18 giờ, vượt quota tuần.


In [8]:
if not RUN_BENCHMARK:
    import yaml

    _c = yaml.safe_load(open('configs/method2/biencoder.yaml', encoding='utf-8'))['train']
    print('giữ nguyên YAML:', {k: _c[k] for k in
          ('batch_size', 'mini_batch_size', 'gradient_checkpointing', 'epochs')})
else:
    bench = json.load(open('results/method2/benchmark_biencoder.json', encoding='utf-8'))
    ok = [b for b in bench if b['ok'] and b['hours_per_epoch']]
    assert ok, 'Không case nào chạy được — xem log ở trên'

    # Chọn theo GIỜ/EPOCH, KHÔNG theo s/step: giảm effective batch làm s/step
    # đẹp hẳn lên nhưng số step mỗi epoch tăng đúng bấy nhiêu lần.
    best = min(ok, key=lambda b: b['hours_per_epoch'])
    print('nhanh nhất theo epoch:', best['case']['name'],
          f"{best['hours_per_epoch']:.2f} h/epoch,",
          f"{best['sec_per_step']:.1f} s/step × {best['steps_per_epoch']:,} step,",
          f"peak {best['peak_vram_mb']:.0f} MB")
    print(f"3 epoch ≈ {best['hours_3_epochs']:.1f} h")

    # safe_dump XOÁ SẠCH comment của YAML — chỉ ghi khi cấu hình thật sự đổi,
    # và nhớ khôi phục file từ git sau khi benchmark xong.
    import yaml

    cfg_path = 'configs/method2/biencoder.yaml'
    cfg = yaml.safe_load(open(cfg_path, encoding='utf-8'))
    cfg['train']['batch_size'] = best['case']['batch_size']
    cfg['train']['mini_batch_size'] = best['case']['mini_batch_size']
    cfg['train']['gradient_checkpointing'] = best['case']['grad_checkpointing']
    yaml.safe_dump(cfg, open(cfg_path, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
    print('đã ghi vào', cfg_path)

    if best['peak_vram_mb'] > 9000:
        print('VRAM', f"{best['peak_vram_mb']:.0f} MB đo khi TẮT eval.",
              'Chạy lại case này CÓ eval trước khi tin là an toàn.')
    if best['case']['batch_size'] != 256:
        print('LƯU Ý: effective batch giảm còn', best['case']['batch_size'],
              '— đổi CHẤT LƯỢNG chứ không chỉ tốc độ, phải ghi vào báo cáo.')


giữ nguyên YAML: {'batch_size': 256, 'mini_batch_size': 32, 'gradient_checkpointing': False, 'epochs': 3}


# Full training — Round 2, đúng plan §Phase 2

Run 1 (2 epoch · `n_hard_negatives` 2 · không mining) đã xong. Vòng này trả
về **đúng cấu hình plan**: `epochs` **3**, `n_hard_negatives` **4**, **có**
mine hard negative + Round 2.

Vừa 11 h quota được là nhờ plan §Phase 2 bước 3: Round 2 **train lại từ
base**, không train tiếp từ Round 1. Nên Round 1 chỉ còn vai trò *máy đào
negative* — dùng lại `run01/final` của session trước, tiết kiệm 9.7 h.

| Bước | Ước tính |
|---|---|
| Nạp `run01/final` từ Kaggle Dataset | ~0 |
| Mine top-20 trên 78,435 query × 4,464 tool | ~0.3 h |
| Round 2 từ base: 921 step × ~38 s | **~9.7 h** |
| Index + calibrate + evaluate | ~0.2 h |
| **Tổng** | **~10.2 h** |

**Điều kiện tiên quyết**: `/kaggle/working` không sống qua session, nên phải
tải Output của notebook cũ (`kaggle kernels output <user>/<slug> -p ./out`
hoặc nút Download) rồi upload `out/artifacts/method2/biencoder/run01/final`
thành Kaggle Dataset và Add vào notebook. Cell dưới tự dò; không thấy thì train
lại Round 1 — và khi đó **không đủ 11 h**, hãy dừng lại.

Còn lệch plan đúng một chỗ: `--single-gpu`. Plan §0 ghi 2×T4 nhưng
sentence-transformers dùng DataParallel, mà GradCache gọi model ~384 lần mỗi
step nên phí scatter/gather đẩy 36 → 475 s/step.

OOM thì đặt `gradient_checkpointing: true` rồi chạy lại cell — nó tự resume
từ checkpoint gần nhất (save mỗi 100 step, ~1 h).


## Round 1 — nạp lại checkpoint đã train (KHÔNG train lại)

Round 1 chỉ dùng để xếp hạng negative cho bước mine. Plan §Phase 2 bước 3
train Round 2 **từ base**, nên chất lượng Round 1 không truyền vào model
cuối — không cần train lại cho đúng cấu hình mới.

`CachedMultipleNegativesRankingLoss` (GradCache) cho effective batch 256.
Gradient accumulation **không** thay thế được: nó chỉ chia nhỏ update chứ
không làm tăng số in-batch negative.


In [9]:
import re, shutil, subprocess, sys

RUN = '/kaggle/working/artifacts/method2/biencoder/run01'

# /kaggle/working không sống qua session, nên run01 phải được upload lại
# thành Kaggle Dataset. Nguồn: tab Output của notebook cũ →
#   kaggle kernels output <user>/<slug> -p ./out
# rồi upload đúng thư mục out/artifacts/method2/biencoder/run01/final.
# Dò theo marker nên không phải hardcode tên dataset.
def find_optional(marker, max_depth=6):
    # depth 6 chứ không phải 4 như Cell 0: dataset chỉ chứa `final/` có
    # thể nằm sâu hơn tuỳ cách zip khi upload.
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT, max_depth):
        if (d / marker).exists():
            return d
    return None


if not Path(f'{RUN}/final/modules.json').exists():
    found = find_optional('final/modules.json')
    if found:
        shutil.copytree(found / 'final', f'{RUN}/final', dirs_exist_ok=True)
        print('nạp Round 1 đã train sẵn từ:', found)

HAVE_ROUND1 = Path(f'{RUN}/final/modules.json').exists()
print('Round 1:', 'CÓ SẴN — bỏ qua train' if HAVE_ROUND1 else 'CHƯA CÓ — sẽ train ~9.7 h')


nạp Round 1 đã train sẵn từ: /kaggle/input/datasets/dathq12/toolcalling-vi-run01
Round 1: CÓ SẴN — bỏ qua train


In [10]:
# Chỉ chạy khi KHÔNG tìm thấy run01 đã train. Với 11 h quota thì nhánh này
# ăn hết ngân sách và Round 2 không còn chỗ — thấy nó chạy thì nên dừng.
if HAVE_ROUND1:
    print('bỏ qua: đã có run01/final')
else:
    # sorted() theo tên là sai: 'checkpoint-1000' < 'checkpoint-500'.
    ckpts = sorted(
        glob.glob(f'{RUN}/checkpoint-*'),
        key=lambda p: int(re.search(r'(\d+)$', p).group(1)),
    )
    resume = ckpts[-1] if ckpts else None
    print('resume from:', resume or '(train từ đầu)')
    # Không nội suy biến có thể None vào chuỗi lệnh: 'None' bị HF coi là
    # đường dẫn checkpoint rồi đi tải từ Hub và chết vì đang chạy offline.
    cmd = [sys.executable, '-m', 'src.models.biencoder.train', 'train',
           '--config', 'configs/method2/biencoder.yaml',
           '--output-dir', RUN, '--single-gpu']
    if resume:
        cmd += ['--resume-from', resume]
    subprocess.run(cmd, check=True)


bỏ qua: đã có run01/final


## Round 2 — mine hard negatives rồi train lại **từ base**

Lấy tool sai nhưng xếp hạng cao (bỏ top-1 để tránh false negative). Train
lại từ checkpoint gốc, **không** train tiếp từ Round 1.

Đây là bước 2–3 của plan §Phase 2 và là **phần chính** của lần chạy này:
3 epoch × `n_hard_negatives` 4 = 921 step ≈ 9.7 h. Mine chỉ chạy một lần —
`train_mined.jsonl` đã có thì bỏ qua, nên chạy lại cell sau khi đứt session
sẽ resume thẳng vào train.


In [11]:
import yaml

_cfg = yaml.safe_load(open('configs/method2/biencoder.yaml', encoding='utf-8'))
RUN_ROUND2 = bool(_cfg.get('mining', {}).get('enabled', False))
RUN2 = '/kaggle/working/artifacts/method2/biencoder/run02'
MINED = _cfg['mining']['output_path']

# subprocess thay vì `!` trong `if`: exit code hiện ra rõ ràng, và một lệnh
# hỏng không bị trôi qua trong Save & Run All.
if RUN_ROUND2:
    assert Path(f'{RUN}/final').exists(), 'thiếu Round 1 để mine'
    # Mine đọc train_path trong YAML, mà bước dưới ghi đè YAML sang
    # train_mined.jsonl — nên phải bỏ qua khi file đã có, nếu không lần chạy
    # thứ hai sẽ mine trên chính output của lần đầu.
    if Path(MINED).exists():
        print('đã có', MINED, '— bỏ qua mine')
    else:
        subprocess.run(
            [sys.executable, '-m', 'src.models.biencoder.train', 'mine',
             '--config', 'configs/method2/biencoder.yaml', '--model', f'{RUN}/final'],
            check=True,
        )
    cfg_text = open('configs/method2/biencoder.yaml', encoding='utf-8').read()
    open('configs/method2/biencoder.yaml', 'w', encoding='utf-8').write(
        cfg_text.replace('biencoder/train.jsonl', 'biencoder/train_mined.jsonl')
    )
    ck2 = sorted(
        glob.glob(f'{RUN2}/checkpoint-*'),
        key=lambda p: int(re.search(r'(\d+)$', p).group(1)),
    )
    cmd = [sys.executable, '-m', 'src.models.biencoder.train', 'train',
           '--config', 'configs/method2/biencoder.yaml',
           '--output-dir', RUN2, '--single-gpu']
    if ck2:
        print('resume Round 2 từ:', ck2[-1])
        cmd += ['--resume-from', ck2[-1]]
    subprocess.run(cmd, check=True)
else:
    print('Round 2 TẮT (mining.enabled=false) — dùng checkpoint của Round 1.')

# Mọi bước sau dùng FINAL, không trỏ cứng vào RUN2: bỏ Round 2 thì RUN2
# không tồn tại và index/calibrate sẽ chết vì không tìm thấy model.
FINAL = RUN2 if RUN_ROUND2 else RUN

# Hết quota giữa chừng thì `final/` không được ghi và mọi cell sau đều
# chết. Checkpoint của SentenceTransformerTrainer nạp được như một model
# hoàn chỉnh, nên rơi về checkpoint mới nhất để vẫn index/eval được.
MODEL = f'{FINAL}/final'
if not Path(MODEL).exists():
    _ck = sorted(glob.glob(f'{FINAL}/checkpoint-*'),
                 key=lambda p: int(re.search(r'(\d+)$', p).group(1)))
    assert _ck, f'không có model nào trong {FINAL}'
    MODEL = _ck[-1]
    print('CHƯA có final/ — train chưa chạy hết, dùng', MODEL)
print('model dùng cho các bước sau:', MODEL)


[biencoder] TensorFlow version 2.20.0 available.
[biencoder] JAX version 0.7.2 available.
[biencoder] No device provided, using cuda:0
[biencoder] Loading SentenceTransformer model from /kaggle/working/artifacts/method2/biencoder/run01/final.
Batches: 100%|██████████| 1226/1226 [14:57<00:00,  1.37it/s]


{
  "n_rows": 94634,
  "n_mined": 78435,
  "output": "data/method2/biencoder/train_mined.jsonl"
}


[biencoder] CUDA_VISIBLE_DEVICES=0 — ép chạy một GPU
[biencoder] TensorFlow version 2.20.0 available.
[biencoder] JAX version 0.7.2 available.
/kaggle/working/src/models/biencoder/train.py:220: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
[biencoder] train pairs: 78435
[biencoder] No device provided, using cuda:0
[biencoder] Loading SentenceTransformer model from BAAI/bge-m3.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8547.00it/s]
[biencoder] The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
/kaggle/working/src/models/biencoder/trai

{'loss': '3.905', 'grad_norm': '2.858', 'learning_rate': '1.054e-05', 'epoch': '0.1629'}
{'loss': '3.035', 'grad_norm': '1.823', 'learning_rate': '2e-05', 'epoch': '0.3257'}


 22%|██▏       | 200/921 [2:17:49<8:21:02, 41.70s/it][biencoder] Saving model checkpoint to /kaggle/working/artifacts/method2/biencoder/run02/checkpoint-200
[biencoder] Saving model to /kaggle/working/artifacts/method2/biencoder/run02/checkpoint-200
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:438: UserWarning: Could not find a config file in BAAI/bge-m3 - will assume that the vocabulary was not modified.
  warnings.warn(

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 18.07it/s]


...

{'eval_custom_val_cosine_accuracy@1': '0.5719', 'eval_custom_val_cosine_accuracy@3': '0.8029', 'eval_custom_val_cosine_accuracy@5': '0.8648', 'eval_custom_val_cosine_accuracy@10': '0.9175', 'eval_custom_val_cosine_precision@1': '0.5719', 'eval_custom_val_cosine_precision@3': '0.2676', 'eval_custom_val_cosine_precision@5': '0.173', 'eval_custom_val_cosine_precision@10': '0.09175', 'eval_custom_val_cosine_recall@1': '0.5719', 'eval_custom_val_cosine_recall@3': '0.8029', 'eval_custom_val_cosine_recall@5': '0.8648', 'eval_custom_val_cosine_recall@10': '0.9175', 'eval_custom_val_cosine_ndcg@10': '0.7519', 'eval_custom_val_cosine_mrr@10': '0.698', 'eval_custom_val_cosine_map@100': '0.7015', 'eval_runtime': '50.13', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '3'}


100%|██████████| 921/921 [10:38:48<00:00, 41.62s/it]
[biencoder] Saving model to /kaggle/working/artifacts/method2/biencoder/run02/final
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:438: UserWarning: Could not find a config file in BAAI/bge-m3 - will assume that the vocabulary was not modified.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 19.28it/s]


{'train_runtime': '3.833e+04', 'train_samples_per_second': '6.139', 'train_steps_per_second': '0.024', 'train_loss': '2.085', 'epoch': '3'}


[biencoder] Information Retrieval Evaluation of the model on the custom_val dataset:
Corpus Chunks: 100%|██████████| 1/1 [00:15<00:00, 15.59s/it]
[biencoder] Queries: 9846
[biencoder] Corpus: 4464

[biencoder] Score-Function: cosine
[biencoder] Accuracy@1: 57.19%
[biencoder] Accuracy@3: 80.29%
[biencoder] Accuracy@5: 86.48%
[biencoder] Accuracy@10: 91.75%
[biencoder] Precision@1: 57.19%
[biencoder] Precision@3: 26.76%
[biencoder] Precision@5: 17.30%
[biencoder] Precision@10: 9.18%
[biencoder] Recall@1: 57.19%
[biencoder] Recall@3: 80.29%
[biencoder] Recall@5: 86.48%
[biencoder] Recall@10: 91.75%
[biencoder] MRR@10: 0.6980
[biencoder] NDCG@10: 0.7519
[biencoder] MAP@100: 0.7015


{
  "custom_val_cosine_accuracy@1": 0.5719073735527117,
  "custom_val_cosine_accuracy@3": 0.8028641072516758,
  "custom_val_cosine_accuracy@5": 0.8648182002843794,
  "custom_val_cosine_accuracy@10": 0.9175299614056469,
  "custom_val_cosine_precision@1": 0.5719073735527117,
  "custom_val_cosine_precision@3": 0.26762136908389195,
  "custom_val_cosine_precision@5": 0.1729636400568759,
  "custom_val_cosine_precision@10": 0.0917529961405647,
  "custom_val_cosine_recall@1": 0.5719073735527117,
  "custom_val_cosine_recall@3": 0.8028641072516758,
  "custom_val_cosine_recall@5": 0.8648182002843794,
  "custom_val_cosine_recall@10": 0.9175299614056469,
  "custom_val_cosine_ndcg@10": 0.7518947027485278,
  "custom_val_cosine_mrr@10": 0.6979559824471402,
  "custom_val_cosine_map@100": 0.7015058299327156
}
{
  "metric": null,
  "resolved_metric": "eval_custom_val_cosine_ndcg@10",
  "best_value": 0.751945670143287,
  "best_step": 900,
  "n_evaluations": 4,
  "available_metrics": [
    "eval_custom_val

## Pre-compute index + hiệu chỉnh ngưỡng trên **val**


In [12]:
!python -m src.models.biencoder.index \
    --config configs/method2/biencoder.yaml --model {MODEL}

# τ và τ_call CHỈ được hiệu chỉnh trên val, rồi freeze trước khi chạy test.
!python -m src.models.biencoder.evaluate calibrate \
    --config configs/method2/biencoder.yaml \
    --model {MODEL} \
    --pairs data/method2/biencoder/val.jsonl \
    --output {FINAL}/thresholds.json


Batches: 100%|██████████████████████████████████| 70/70 [00:49<00:00,  1.41it/s]
[index] 4464 tool × 1024d trong 49.705s → data/method2/index/tool_embeddings.npy
Loading weights: 100%|██████████████████████| 290/290 [00:00<00:00, 2265.95it/s]
{
  "tau": 0.35,
  "tau_call": 0.34,
  "k_max": 3,
  "gap_delta": 0.2,
  "strategy": "absolute",
  "calibrated_on": "data/method2/biencoder/val.jsonl",
  "metrics": {
    "abstention": {
      "macro_f1": 0.9152,
      "f1_call": 0.982,
      "f1_no_call": 0.8485,
      "negative_recall": 0.874,
      "call_recall": 0.9786
    },
    "call_selection": {
      "absolute": {
        "value": 0.34,
        "f1": 0.9475,
        "precision": 0.94,
        "recall": 0.955,
        "tool_set_accuracy": 0.8802
      },
      "gap": {
        "value": 0.2,
        "f1": 0.9543,
        "precision": 0.9546,
        "recall": 0.954,
        "tool_set_accuracy": 0.8947
      },
      "winner": "gap"
    },
    "retrieval": {
      "n_positive_samples": 7521,

## Gate để qua Phase 3

| Metric | Tập | Ngưỡng |
|---|---|---|
| Recall@1 | custom `val_seen` | ≥ 0.90 |
| Recall@1 | custom `val_unseen` | ≥ 0.75 |
| Recall@5 | custom `val_unseen` | ≥ 0.92 |
| Negative Recall @ τ | custom val negative | ≥ 0.80 |

Không đạt → thử theo thứ tự: (a) thêm param name vào document text,
(b) tăng hard negative lên 8, (c) đổi sang `AITeamVN/Vietnamese_Embedding`,
(d) full fine-tune thay LoRA.


In [13]:
!python -m src.models.biencoder.evaluate evaluate \
    --config configs/method2/biencoder.yaml \
    --model {MODEL} \
    --pairs data/method2/biencoder/val.jsonl \
    --output results/method2/metrics/biencoder_val.json

report = json.load(open('results/method2/metrics/biencoder_val.json', encoding='utf-8'))
for slice_name, metrics in report['by_source_key'].items():
    print(slice_name, {k: v for k, v in metrics.items() if 'recall@' in k or k == 'mrr'})


Loading weights: 100%|██████████████████████| 290/290 [00:00<00:00, 2235.31it/s]
{
  "n_positive_samples": 7521,
  "n_gold_tools": 9846,
  "mrr": 0.9968,
  "ndcg@10": 0.9969,
  "micro_recall@1": 0.7599,
  "full_recall@1": 0.725,
  "micro_recall@3": 0.9936,
  "full_recall@3": 0.9918,
  "micro_recall@5": 0.9999,
  "full_recall@5": 0.9999,
  "micro_recall@10": 1.0,
  "full_recall@10": 1.0
}
benchmark_val {'mrr': 0.9966, 'micro_recall@1': 0.7545, 'full_recall@1': 0.718, 'micro_recall@3': 0.9934, 'full_recall@3': 0.9914, 'micro_recall@5': 0.9999, 'full_recall@5': 0.9999, 'micro_recall@10': 1.0, 'full_recall@10': 1.0}
custom_val_seen {'mrr': 1.0, 'micro_recall@1': 0.8696, 'full_recall@1': 0.85, 'micro_recall@3': 1.0, 'full_recall@3': 1.0, 'micro_recall@5': 1.0, 'full_recall@5': 1.0, 'micro_recall@10': 1.0, 'full_recall@10': 1.0}
custom_val_unseen {'mrr': 1.0, 'micro_recall@1': 0.8696, 'full_recall@1': 0.85, 'micro_recall@3': 0.9957, 'full_recall@3': 0.995, 'micro_recall@5': 1.0, 'full_recall

### Số pool-scope — con số đưa vào luận văn

Cell trên chạy `--scope candidates` (mặc định): mỗi query chỉ xếp hạng
trong candidate pool của chính nó — benchmark 1–8 tool, custom đúng 10.
Ở đó `recall@10 = 1.0` là **tất yếu**, không phải thành tích, và
`micro_recall@1` bị chặn trên bởi `n_sample / n_gold` (sample 2 gold tool
ở k=1 chỉ lấy được 1).

`--scope pool` xếp hạng trên cả 4,464 tool. Tốn ~3 phút, và đây mới là
thiết lập so sánh được với Method 1.


In [14]:
!python -m src.models.biencoder.evaluate evaluate \
    --config configs/method2/biencoder.yaml \
    --model {MODEL} \
    --pairs data/method2/biencoder/val.jsonl \
    --scope pool \
    --output results/method2/metrics/biencoder_val_pool.json

pool = json.load(open('results/method2/metrics/biencoder_val_pool.json', encoding='utf-8'))
cand = report['overall']
print(f"{'slice':22s} {'R@1 cand':>9s} {'R@1 pool':>9s}")
print(f"{'TỔNG':22s} {cand['micro_recall@1']:9.4f} {pool['overall']['micro_recall@1']:9.4f}")
for name, m in pool['by_source_key'].items():
    if 'micro_recall@1' not in m:
        continue  # slice toàn negative, không có metric truy hồi
    c = report['by_source_key'][name].get('micro_recall@1')
    print(f"{name:22s} {c:9.4f} {m['micro_recall@1']:9.4f}")


Loading weights: 100%|██████████████████████| 290/290 [00:00<00:00, 2138.44it/s]
{
  "n_positive_samples": 7521,
  "n_gold_tools": 9846,
  "mrr": 0.833,
  "ndcg@10": 0.836,
  "micro_recall@1": 0.5719,
  "full_recall@1": 0.5332,
  "micro_recall@3": 0.8029,
  "full_recall@3": 0.7796,
  "micro_recall@5": 0.8644,
  "full_recall@5": 0.8446,
  "micro_recall@10": 0.9176,
  "full_recall@10": 0.9024
}
slice                   R@1 cand  R@1 pool
TỔNG                      0.7599    0.5719
benchmark_val             0.7545    0.5599
custom_val_seen           0.8696    0.8696
custom_val_unseen         0.8696    0.7652


## Run manifest — chốt lại toàn bộ mục audit

Train xong mà không audit được thì coi như chưa train. Cell này gom: commit
SHA (kèm cờ dirty), config YAML thực tế, fingerprint dataset + tool pool,
query counts và overlap theo split, số positive/negative pair, checkpoint,
best step + metric đã dùng để chọn, VRAM peak, thời lượng train, và
Recall@1/@5/@10 + MRR. Thiếu mục nào thì `audit_complete.missing` liệt kê ra.

`checkpoint_selection.available_metrics` cho biết tên metric thật của
`InformationRetrievalEvaluator` ở phiên bản đang chạy — khai vào
`train.metric_for_best_model` cho lần chạy sau.


In [15]:
!python -m src.models.run_manifest \
    --run-dir {FINAL} \
    --config configs/method2/biencoder.yaml \
    --stage biencoder \
    --report retrieval=results/method2/metrics/biencoder_val.json

manifest = json.load(open(f'{FINAL}/run_manifest.json', encoding='utf-8'))
missing = manifest['audit_complete']['missing']
print('thiếu:', missing or 'không thiếu mục nào')
print('Recall/MRR:', manifest.get('retrieval_gate', {}).get('metrics'))
print('VRAM peak MB:', manifest['train']['peak_vram_mb'])
print('thời lượng (giờ):', manifest['train']['duration_hours'])
print('chọn checkpoint:', manifest['train']['checkpoint_selection'])
assert not missing, f'Chưa đủ artifact để audit: {missing}'


[run_manifest] → /kaggle/working/artifacts/method2/biencoder/run02/run_manifest.json
[run_manifest] commit=6eb1b582b4eaba9ff8a9b51c4d88ae5bd0898465 dirty=False
[run_manifest] overlap sau decontamination: {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[run_manifest] OK: đủ toàn bộ mục audit
thiếu: không thiếu mục nào
Recall/MRR: {'micro_recall@1': 0.7599, 'micro_recall@5': 0.9999, 'micro_recall@10': 1.0, 'mrr': 0.9968}
VRAM peak MB: 10888.9
thời lượng (giờ): 10.647
chọn checkpoint: {'metric': None, 'resolved_metric': 'eval_custom_val_cosine_ndcg@10', 'best_value': 0.751945670143287, 'best_step': 900, 'n_evaluations': 4, 'available_metrics': ['eval_custom_val_cosine_accuracy@1', 'eval_custom_val_cosine_accuracy@10', 'eval_custom_val_cosine_accuracy@3', 'eval_custom_val_cosine_accuracy@5', 'eval_custom_val_cosine_map@100', 'eval_custom_val_cosine_mrr@10', 'eval_custom_val_cosine_ndcg@10', 'eval_custom_val_cosine_precision@1', 'eval_custom_val_cosine_precision@10', 'eval_custom_val_cosin

In [16]:
# ===== Lưu artifact =====
# Kaggle giữ /kaggle/working trong Output của version; tar chỉ là tiện lợi. Gói cả results/ vì metric JSON cũng là artefact audit.
!tar czf /kaggle/working/biencoder_run.tar.gz \
    -C /kaggle/working artifacts/method2 results/method2
!du -h /kaggle/working/biencoder_run.tar.gz


216M	/kaggle/working/biencoder_run.tar.gz
